In [1]:
from check_splice import config, get_cpcdh

cfg = config.pcdh()
df = get_cpcdh(cfg["data_dir"] / "hg19.ncbiRefSeq.gtf.gz")
df.to_csv(cfg["data_dir"] / "cpcdh.csv", index=False)

In [2]:
from check_splice import config, pair_cpcdh

cfg = config.pcdh()
df = pair_cpcdh(cfg["data_dir"] / "cpcdh.csv")
df.to_csv(cfg["data_dir"] / "cpcdh_pair.csv", index=False)

In [ ]:
from check_splice import config, parse_chimeric

cfg = config.pcdh()
for samfile in [
    "total_rna_seq/WT.bam",
    "rna_seq/RNA-seq-PPdel-PPHLN1.merge.bam",
    "rna_seq/RNA-seq-WT-PPHLN1.merge.bam",
    "rna_seq/TASORdel.bam",
]:
    df = parse_chimeric(
        samfile=cfg["data_dir"] / samfile,
        locus_chrom=cfg["chrom"],
        locus_start=cfg["start"],
        locus_end=cfg["end"],
    )
    print(df["is_chimeric"].value_counts())
    df.to_csv(cfg["data_dir"] / f"{samfile}.chimeric", index=False)

In [ ]:
from check_splice import config, nearby_explode

cfg = config.pcdh()
for chimeric_file in [
    "total_rna_seq/WT.bam.chimeric",
    "rna_seq/RNA-seq-PPdel-PPHLN1.merge.bam.chimeric",
    "rna_seq/RNA-seq-WT-PPHLN1.merge.bam.chimeric",
    "rna_seq/TASORdel.bam.chimeric",
]:
    df = nearby_explode(cfg["data_dir"] / chimeric_file)
    df.to_csv(cfg["data_dir"] / f"{chimeric_file}.nearby", index=False)

In [3]:
import pandas as pd

from check_splice import config, count_splice

cfg = config.pcdh()
cpcdh_pair = pd.read_csv(cfg["data_dir"] / "cpcdh_pair.csv")
for nearby_file in [
    "total_rna_seq/WT.bam.chimeric.nearby",
    "rna_seq/RNA-seq-PPdel-PPHLN1.merge.bam.chimeric.nearby",
    "rna_seq/RNA-seq-WT-PPHLN1.merge.bam.chimeric.nearby",
    "rna_seq/TASORdel.bam.chimeric.nearby",
]:
    df_nearby = pd.read_csv(cfg["data_dir"] / nearby_file, header=0)
    counts = []
    for seqname1, end1, strand1, seqname2, start2, strand2 in zip(
        cpcdh_pair["seqname1"],
        cpcdh_pair["end1"],
        cpcdh_pair["strand1"],
        cpcdh_pair["seqname2"],
        cpcdh_pair["start2"],
        cpcdh_pair["strand2"],
    ):
        count = count_splice(
            df_nearby,
            splice1_chrom=seqname1,
            splice1_pos=end1,
            splice1_strand=strand1,
            splice2_chrom=seqname2,
            splice2_pos=start2,
            splice2_strand=strand2,
            splice_thres=cfg["splice_thres"],
        )
        counts.append(count)

    cpcdh_pair.assign(
        count=counts,
        size1=lambda df: df["end1"] - df["start1"],
        size2=lambda df: df["end2"] - df["start2"],
    ).to_csv(cfg["data_dir"] / f"{nearby_file}.splice", index=False)

In [ ]:
import pandas as pd

from check_splice import config, count_non_splice

cfg = config.pcdh()
cpcdh = pd.read_csv(cfg["data_dir"] / "cpcdh.csv")
for nearby_file in [
    "total_rna_seq/WT.bam.chimeric.nearby",
    "rna_seq/RNA-seq-PPdel-PPHLN1.merge.bam.chimeric.nearby",
    "rna_seq/RNA-seq-WT-PPHLN1.merge.bam.chimeric.nearby",
    "rna_seq/TASORdel.bam.chimeric.nearby",
]:
    df_nearby = pd.read_csv(cfg["data_dir"] / nearby_file, header=0)
    start_counts = []
    end_counts = []
    for seqname, start, end, strand in zip(
        cpcdh["seqname"],
        cpcdh["start"],
        cpcdh["end"],
        cpcdh["strand"],
    ):
        start_count = count_non_splice(
            df_nearby,
            splice_chrom=seqname,
            splice_pos=start,
            splice_strand=strand,
            non_splice_thres=cfg["non_splice_thres"],
        )
        start_counts.append(start_count)
        end_count = count_non_splice(
            df_nearby,
            splice_chrom=seqname,
            splice_pos=end,
            splice_strand=strand,
            non_splice_thres=cfg["non_splice_thres"],
        )
        end_counts.append(end_count)

    cpcdh.assign(
        start_count=start_counts,
        end_count=end_counts,
        size=lambda df: df["end"] - df["start"],
    ).to_csv(cfg["data_dir"] / f"{nearby_file}.non_splice", index=False)